# JSON-to-RDF (Only Entity) Converter

In [16]:
import json

def _assign_entity_ids(data):

    # load training data from json
    # with open(file, 'r', encoding='utf-8') as file:
    #     data = json.load(file)
    
    # check whether this is an initial annotation
    # a null entity id with Candidate status indicates an initial annotation
    is_candidate_id_null = False
    
    for i in data['annotations']:
        for ent in i[2]['entities']:
            entity_id = ent[0]
    
            if entity_id is None:
                for sts in ent[3]:
                    status = sts[0]
                    
                    if status == 'Candidate':
                        is_candidate_id_null = True
                        
                    break
            
            if is_candidate_id_null:
                break
        
        if is_candidate_id_null:
            break
    
    # assign entity ids for the initial annotation
    if is_candidate_id_null:
        for i in data['annotations']:
            paragraph_id = i[0]
            entity_number = 1
            
            for ent in i[2]['entities']:
                entity_id = f'{paragraph_id}_E{entity_number}'
                ent[0] = entity_id
                entity_number += 1

    if not is_candidate_id_null: 
        # determine the current review number from the ids of the suggested entities (span only) in the previous review
        review_numbers = []
        
        for i in data['annotations']:
            for ent in i[2]['entities']:
                if ent[0] is not None:
                    id_last_part = ent[0].split('_')[-1]
            
                    if id_last_part[0:2] == 'Rv':
                        review_numbers.append(int(id_last_part[2:]))
        
        if bool(review_numbers):
            largest_number = max(review_numbers)
            id_last_part = f'Rv{largest_number+1}'
        else:
            id_last_part = 'Rv1'
    
        # assign entity ids to entities added during the review
        for i in data['annotations']:
            paragraph_id = i[0]
            entity_number = 1
            
            for ent in i[2]['entities']:
                if ent[0] is None:
                    entity_id = f'{paragraph_id}_E{entity_number}_{id_last_part}'
                    ent[0] = entity_id
                    entity_number += 1

    return data


In [17]:
def _get_prefixes():
    prefixes = [
        'PREFIX onner: <http://purl.org/spatialai/onner/onner-full#>',
        'PREFIX data: <http://purl.org/spatialai/onner/onner-full/data#>',
        'PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>',
        'PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>',
        'PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>',
        'PREFIX owl: <http://www.w3.org/2002/07/owl#>',
    ]

    return '\n'.join(prefixes) + '\n'

In [18]:
def generate_entity_rdf_anner(file, labeling_schema):

    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    data = _assign_entity_ids(data)
    
    xsd_date = '^^xsd:date'
    xsd_string = '^^xsd:string'
    xsd_non_neg_int = '^^xsd:nonNegativeInteger'
        
    rdf = ''
    labels_found = {}
    annotators = {
        'model': [],
        'human': []
    }

    schema_name = labeling_schema['name']
    schema_labels = labeling_schema['labels']
    
    for paragraph in data['annotations']:
        paragraph_id = paragraph[0]
        paragraph_text = paragraph[1]
        entity_ids = []
    
        # join entity ids in a single string
        for entity in paragraph[2]['entities']:
            entity_id = entity[0]
            entity_ids.append(f'data:{entity_id}')

        if bool(entity_ids):
            entity_ids_joined = ', '.join(entity_ids)
            
            rdf += f"data:{paragraph_id} onner:directlyContainsLabeledTerm {entity_ids_joined} .\n\n"
        
            # access all information of annotations
            for entity in paragraph[2]['entities']:
                entity_id = entity[0]
                start = int(entity[1])
                end = int(entity[2])
                entity_text = paragraph_text[start:end]
                entity_length = len(entity_text)
                status_ids = []
                
                rdf += f"data:{entity[0]} rdf:type onner:LabeledTerm ;\n"    # deal with atomic and compound terms
                rdf += f"onner:labeledTermText '{entity_text}'{xsd_string} ;\n"
                rdf += f"onner:offset '{start}'{xsd_non_neg_int} ;\n"
                rdf += f"onner:length '{entity_length}'{xsd_non_neg_int} ;\n"
                rdf += f"onner:labeledTermDirectlyContainedBy data:{paragraph_id} ;\n"
        
                # join all status ids of an annotation in a single string
                for status_block in entity[3]:
                    status = status_block[0]
                    status_date = status_block[2]
                    formatted_date = status_date[2:16].replace('-', '').replace(':', '') + 'Z'
                    status_id = f'{status}_{formatted_date}_{entity_id}'
                    status_ids.append(f'data:{status_id}')
        
                status_ids_joined = ', '.join(status_ids)
                
                rdf += f"onner:hasLabeledTermStatus {status_ids_joined} .\n\n"
        
                # access the status block
                for status_block in entity[3]:
                    status = status_block[0]
                    status_date = status_block[2]
                    formatted_date = status_date[2:16].replace('-', '').replace(':', '') + 'Z'
                    status_id = f'{status}_{formatted_date}_{entity_id}'
                    status_assigner = status_block[3]
                    label = status_block[1]
                    label_number = schema_labels[label]
        
                    # store all unique labels found in a file
                    if label not in labels_found:
                        labels_found.update({label: label_number})
        
                    # store all unique annotator found in a file
                    if 'Model_' in status_assigner:
                        if status_assigner not in annotators['model']:
                            annotators['model'].append(status_assigner)
                    else:
                        if status_assigner not in annotators['human']:
                            annotators['human'].append(status_assigner)
        
                    rdf += f"data:{status_id} rdf:type onner:{status}Status ;\n"
                    rdf += f"onner:statusAssignmentDate '{status_date}'{xsd_date} ;\n"
                    rdf += f"onner:statusAssignedBy data:{status_assigner} ;\n"
                    rdf += f"onner:hasLabeledTermLabel data:{schema_name}_Label{label_number} .\n\n"

        else:    # check this condition with zero-entity paragraphs
            rdf += f"data:{paragraph_id} onner:directlyContainsLabeledTerm data:NoLabeledTerm .\n\n"
            
    for key, value in labels_found.items():
        rdf += f"data:{schema_name}_Label{value} rdf:type onner:Label ;\n"
        rdf += f"onner:fromLabelingSchema data:Labeling_Schema ;\n"
        rdf += f"onner:labelText '{key}'{xsd_string} .\n\n"
    
    rdf += f"data:{schema_name}_Labels rdf:type onner:LabelingSchema ;\n"
    rdf += f"onner:schemaName '{schema_name}'{xsd_string} .\n\n"
    
    for model in annotators['model']:
        version = model.split('_')[-1]
        rdf += f"data:{model} rdf:type onner:NER_System ;\n"
        rdf += f"onner:systemVersion '{version}'{xsd_string} .\n\n"
    
    for name in annotators['human']:
        rdf += f"data:Person_{name} rdf:type onner:Person ;\n"
        rdf += f"onner:personName '{name}'{xsd_string} .\n\n"
    
    return rdf



In [24]:
# test cell: function's output check
file = '/home/umayer/Work/research/ner_data/annotation_reviewed/Khoshkava_2014/Khoshkava_2014_merged.json'
labeling_schema = {
    'name': 'CelloGraph',
    'labels': {
        'CHEM_ENT':         1, 
        'MAT_ENT_STRUCT':   2, 
        'MAT_ENT_UNSTRUCT': 3,
        'PROPERTY':         4,
        'END_USE':          5,
        'PROCESS':          6,
        'EQUIPMENT':        7,
        'MEASUREMENT':      8,
        'ABBREVIATION':     9        
    }
}

rdf = generate_entity_rdf_anner(file, labeling_schema)
print(_get_prefixes())
print(rdf)

PREFIX onner: <http://purl.org/spatialai/onner/onner-full#>
PREFIX data: <http://purl.org/spatialai/onner/onner-full/data#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

data:10.1016_j.powtec.2014.04.016_A_P1 onner:directlyContainsLabeledTerm data:10.1016_j.powtec.2014.04.016_A_P1_E1, data:10.1016_j.powtec.2014.04.016_A_P1_E2, data:10.1016_j.powtec.2014.04.016_A_P1_E3, data:10.1016_j.powtec.2014.04.016_A_P1_E1_Rv2, data:10.1016_j.powtec.2014.04.016_A_P1_E4, data:10.1016_j.powtec.2014.04.016_A_P1_E5, data:10.1016_j.powtec.2014.04.016_A_P1_E6, data:10.1016_j.powtec.2014.04.016_A_P1_E7, data:10.1016_j.powtec.2014.04.016_A_P1_E8, data:10.1016_j.powtec.2014.04.016_A_P1_E9, data:10.1016_j.powtec.2014.04.016_A_P1_E10, data:10.1016_j.powtec.2014.04.016_A_P1_E11, data:10.1016_j.powtec.2014.04.016_A_P1_E12, data:10.1016_j.powtec.2014.04.01